# 08 -- Promotion / rejection memo

The final, single-use out-of-sample check and the written decision. The lockbox enforces that the held-out slice is touched exactly once.

In [ ]:
# Parameters (papermill-overridable: `papermill ... -p SYMBOLS '["AAPL","MSFT"]'`)
SYMBOLS = ["ALPHA", "BRAVO", "CHARLIE"]
START = "2018-01-01"
N_DAYS = 600
SEED = 7
USE_SYNTHETIC = True  # set False to fetch real data via core_trading.data.sources


In [ ]:
import numpy as np
import pandas as pd

from core_trading.research.reproducibility import set_seeds

set_seeds(SEED)


def synthetic_bars(symbols, n, seed, start=START):
    """Seeded OHLCV frame in the canonical (symbol, timestamp) layout."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=n, freq="B", tz="UTC")
    frames = []
    for k, sym in enumerate(symbols):
        drift = 0.0003 * (1 + k)
        px = 100.0 + np.cumsum(rng.standard_normal(n) + drift)
        px = np.maximum(px, 1.0)
        high = px + np.abs(rng.standard_normal(n)) * 0.4
        low = px - np.abs(rng.standard_normal(n)) * 0.4
        frame = pd.DataFrame(
            {
                "open": px,
                "high": np.maximum(high, px),
                "low": np.minimum(low, px),
                "close": px,
                "volume": rng.uniform(1e6, 5e6, n),
                "source": "synthetic",
            },
            index=pd.MultiIndex.from_product(
                [[sym], idx], names=["symbol", "timestamp"]
            ),
        )
        frames.append(frame)
    return pd.concat(frames).sort_index()


if USE_SYNTHETIC:
    bars = synthetic_bars(SYMBOLS, N_DAYS, SEED)
else:  # pragma: no cover - exercised only against live vendors
    import asyncio

    from core_trading.data.bars import BarRequest, BarResolution
    from core_trading.data.sources.yfinance_source import YFinanceBarSource

    req = BarRequest(
        symbols=tuple(SYMBOLS),
        resolution=BarResolution.DAY_1,
        start=pd.Timestamp(START, tz="UTC").to_pydatetime(),
        end=pd.Timestamp.now(tz="UTC").to_pydatetime(),
    )
    bars = asyncio.run(YFinanceBarSource().fetch_bars(req))

print(f"loaded {bars.shape[0]} bars across {len(SYMBOLS)} symbols")
bars.head()


In [ ]:
from pathlib import Path

from core_trading.research.feature_store import default_feature_store
from core_trading.research.overfitting import (
    OutOfSampleLockbox, sharpe_ratio)

store = default_feature_store()
z = store.compute(bars, ['zscore_20'])['zscore_20'].unstack('symbol')
rets = bars['close'].unstack('symbol').pct_change()
w = (-z / 2.0).clip(-1.0, 1.0)
g = w.abs().sum(axis=1).replace(0.0, np.nan)
w = w.div(g, axis=0).fillna(0.0)
strat_ret = (w.shift(1) * rets).sum(axis=1).fillna(0.0).to_frame('ret')

In [ ]:
ledger = Path('lockbox_ledger.json')
box = OutOfSampleLockbox(strat_ret, lockbox_fraction=0.2,
                         ledger_path=ledger, name='zscore20_meanrev')
dev_sharpe = sharpe_ratio(box.development_set['ret'])
print(f'development Sharpe: {dev_sharpe:.3f}')

# Touch the lockbox exactly once for the final read-out.
if not box.is_opened():
    holdout = box.open_lockbox('final OOS evaluation of zscore20 mean-reversion')
    oos_sharpe = sharpe_ratio(holdout['ret'])
    print(f'lockbox (OOS) Sharpe: {oos_sharpe:.3f}')
else:
    print('lockbox already opened -- OOS is single-use; see ledger.')

## Decision memo

| Criterion | Threshold | Result |
|---|---|---|
| Deflated Sharpe | >= 0.95 | _fill in_ |
| PBO | < 0.5 | _fill in_ |
| OOS Sharpe vs development | within 1 SE | _fill in_ |

**Decision:** PROMOTE to paper / REJECT (delete). If promoting, the signal graduates into `core_trading/signals/` and enters the Phase 4 paper-trading pipeline. If rejecting, record the reason here so the idea is not silently retried.